In [0]:
%restart_python

In [0]:
print("")   
%pip install nltk
import nltk
nltk.download('stopwords')

In [0]:
### VECTORIZATION **TF-IDF** with library `scikit-learn`

### Código de Vectorización TF-IDF



from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
import pandas as pd
from pathlib import Path
import string
import nltk
import re

# # 1. Load the Dataset 
# from pathlib import Path
# DATA_DIR = Path("../data") #./data Actual level ---- ../data 2 levels up
# # DATA_DIR = Path("/dbfs/workspace/datasets/imdb_big_dataset")
# movies_path = DATA_DIR / "movies_final.csv"

# # 0. Loading datasets imbd and links 
# # Those two datasets are the conection between each movie and its information
# movies_df = pd.read_csv(movies_path)

# Asegúrate de que el catálogo y el esquema sean correctos al leer la tabla
movies_df = pd.read_csv("../data/movies_final.csv")
# movies_df = spark.read.table("workspace.datasets.movies_final").toPandas()
movies_df.head(5)

# 1. Clean stopwords
stopwords = nltk.corpus.stopwords.words('english')
def clean_text(text):
    text = "".join([word for word in text if word not in string.punctuation])
    tokens = re.split('\\W+', text)
    text = [word for word in tokens if word not in stopwords]
    return text

#Apply the function
movies_df['combined_features_nostop'] = movies_df['combined_features'].apply(lambda x: clean_text(x.lower()))
movies_df.head(5)

In [0]:
print(len(movies_df.columns))
movies_df.columns

In [0]:
ps = nltk.PorterStemmer()

def stemming(tokenized_text):
    text = [ps.stem(word) for word in tokenized_text]
    return text

movies_df['combined_features_stemmed'] = movies_df['combined_features_nostop'].apply(lambda x: stemming(x))

movies_df.head(5)

In [0]:
len(movies_df.iloc[0,len(movies_df.columns) - 1])
# print(movies_df.iloc[12])

print("combined_features") 
print(movies_df.iloc[0, 10])

print("\ncombined_features_nostop") #Punctuation
print(movies_df.iloc[0, 11])

print("\ncombined_features_stemmed") #Steemed
print(movies_df.iloc[0, 12])

# movies_df.iloc['combined_features_nostop']
# movies_df.iloc['combined_features']


In [0]:
# Revisar tipos en una columna específica
tipo_por_fila = movies_df["combined_features_stemmed"].map(type)
print(tipo_por_fila.value_counts())

# LIMPIAR LISTAS EN STEEMED PARA PODER VECTORIZAR 

# 3. VECTORIZATION TF-IDF

In [0]:
from sklearn.feature_extraction.text import CountVectorizer

corpus = movies_df["combined_features_stemmed"].fillna("").astype(str)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(corpus)

In [0]:
# # 2. Start Vectorizator TF-IDF with N-gram = 2 (e.g. "science fiction")
#Convierte cada lista de tokens a un string antes de vectorizar:

# Convierte cada lista de tokens a un string antes de llamar a fit_transform.
corpus = movies_df["combined_features_stemmed"] \
    .fillna("") \
    .apply(lambda tokens: " ".join(tokens) if isinstance(tokens, list) else str(tokens))

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=0.01,
    max_df=0.8
)

tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
print(f"Matriz generada. Forma: {tfidf_matrix.shape}")
type(tfidf_matrix)

In [0]:
DATA_DIR1 = Path("../models")
# # # 4. Guardar los artefactos en Databricks para su uso posterior en el chatbot
# # Es crucial guardar tanto la matriz como el vectorizador para procesar las consultas del usuario [3, 4]
# joblib.dump(tfidf_vectorizer, DATA_DIR1 / 'tfidf_vectorizer.pkl')
# joblib.dump(tfidf_matrix, DATA_DIR1 / 'tfidf_matrix.pkl')

#Two ways to save the model ----> 
# 1. Using the path of the folder for the vectorizer
joblib.dump(tfidf_vectorizer, '/Workspace/Users/d.esteban.am@gmail.com/AI-and-ML-Laboratory-Databricks/models/tfidf_vectorizer.pkl')
# 2. Using the path of the file for the matrix
joblib.dump(tfidf_matrix, DATA_DIR1 / 'tfidf_matrix.pkl')
print("Artefactos guardados en la carpeta /models/")
print(type(tfidf_matrix))

# STAGE #4------> COSINE SIMILARITY

In [0]:
# from sklearn.metrics.pairwise import cosine_similarity
# import joblib
# from pathlib import Path
# DATA_DIR = Path("../models") #./data Actual level ---- ../data 2 levels up

# # 1. Calcular la matriz de similitud de coseno
# # Esto comparará cada fila (película) con todas las demás
# similarity_matrix = cosine_similarity(tfidf_matrix)

# # 2. Guardar la matriz para no tener que recalcularla
# # joblib.dump(similarity_matrix, '/dbfs/mnt/tu_ruta/models/similarity_matrix.pkl')
# joblib.dump(similarity_matrix, DATA_DIR / 'similarity_matrix.pkl')

# print(f"Matriz de similitud generada: {similarity_matrix.shape}")